# 公平对比入口（默认 CPU 合成）

这是后续 Colab 实验的入口。仓库里的 `colab_probe.py` 和 `colab_loader_test.py` 只是环境探测，不是本入口。`colab_stage_e_cont.py` 会从 ep10 往下续训，不要拿来跑这四组。

默认不挂载 Drive、不下载权重或医学图像、不占用 GPU。`RUN_SYNTHETIC_TRAIN = True` 会用替代小骨干跑四组冻结选择、梯度累积、2 step 断点恢复和一轮推理。`RUN_REAL = False` 时不会调用真实训练。

把 `RUN_REAL` 改成 True 之后，笔记本调用 `launch_real`：本地权重、本地记下的 sha256、数据根和 manifest。`DEVICE` 保持 `cpu`，除非明确写成 `cuda`（那时要求 bf16）。本文件在自动测试里只在 CPU 上执行过，没有在 Colab GPU 上验证。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "revision").exists():
    raise SystemExit("请把工作目录设为仓库根目录后再跑")
sys.path.insert(0, str(ROOT))
MOUNT_DRIVE = False
RUN_REAL = False
RUN_SYNTHETIC_TRAIN = True
INSTALL_DEPS = False
DEVICE = "cpu"  # 改成 cuda 才会申请 GPU，并且要求 bf16
SEED = 7
SEED_PHASE = "single"  # 改成 triple 才会展开三组 seed
TRAIN_MODE = "frozen_probe"  # 或 last4_finetune
BACKBONE_ID = "innerlogic_surg_vitb16"
WEIGHT_PATH = ""  # 本地权重；留空则真实入口 fail-fast
EXPECTED_SHA = ""  # 在本地记下的 sha256，仓库不预存私有哈希
DATA_ROOT = ""
MANIFEST = ""  # jsonl，含 train / val；test 行不会被读取
OUT_DIR = "/content/ils_fair_runs" if Path("/content").exists() else "/tmp/ils_fair_runs"
print("ROOT", ROOT)
print("OUT_DIR", OUT_DIR)
print("DEVICE", DEVICE)


In [ ]:
import subprocess
import sys
PIN = "pip install -r requirements-fair.txt --extra-index-url https://download.pytorch.org/whl/cpu"
print("工作目录", ROOT)
print("版本钉", (ROOT / "requirements-fair.txt").read_text())
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-fair.txt"), "--extra-index-url", "https://download.pytorch.org/whl/cpu"])
else:
    print("未自动安装。需要时在仓库根目录执行：")
    print(PIN)


In [ ]:
import socket
import shutil
from pathlib import Path
print("hostname", socket.gethostname())
print("同一台 VM 以这个 hostname 为准，不要凭链接判断")
probe = Path(OUT_DIR)
probe.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(probe)
print("path", probe, "exists", probe.exists(), "disk_free_bytes", usage.free)


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive 已按浏览器授权挂上。请再打印一次 hostname，确认和上面是同一台。")
else:
    print("未挂载 Drive。默认不访问私人数据。")


In [ ]:
from revision.preflight import environment_report
from revision.fair_config import runtime_plan, training_matrix
report = environment_report(OUT_DIR)
print(report)
print(runtime_plan(real=RUN_REAL, cuda=report["cuda"], bf16=report["bf16"], device=DEVICE))
print("single-seed runs", len(training_matrix(phase=SEED_PHASE, seed=SEED)))
print("选点", training_matrix()[0]["selection_score"], training_matrix()[0]["selection_protocol"])


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "revision.smoke", "--out", OUT_DIR])
print("smoke 日志", Path(OUT_DIR) / "smoke.log")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from revision.engine import run_synthetic_demo
if RUN_SYNTHETIC_TRAIN:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    synth_root = Path(OUT_DIR) / f"cpu-synth-{stamp}"
    synth_root.mkdir()
    summary = run_synthetic_demo(synth_root, optimizer_steps=2)
    print("synthetic_configs", len(summary["configs"]))
    for item in summary["configs"]:
        print(item["train_mode"], "steps", item["optimizer_steps"], "backward", item["backward_passes"], "probe", item["grad_probe"])
    print("resume_match", summary["resume_match"])
    print("infer", summary["infer"]["ckpt_kind"], summary["infer"]["warnings"], summary["infer"]["out_dir"])
else:
    print("RUN_SYNTHETIC_TRAIN 为 False，跳过合成训练。")


In [ ]:
from revision.engine import launch_real
from revision.identity import RevisionError
if RUN_REAL:
    try:
        result = launch_real(
            backbone_id=BACKBONE_ID,
            weight_path=WEIGHT_PATH,
            expected_sha=EXPECTED_SHA,
            data_root=DATA_ROOT,
            manifest=MANIFEST,
            out_dir=OUT_DIR,
            train_mode=TRAIN_MODE,
            device=DEVICE,
            seed=SEED,
        )
        print("real_run", result["run_dir"], result["optimizer_steps"], result["selection_metric"])
    except RevisionError as exc:
        print(exc.code, exc)
        raise
else:
    print("RUN_REAL 为 False。真实训练或推理要把它改成 True，并填 WEIGHT_PATH、EXPECTED_SHA、DATA_ROOT、MANIFEST。DEVICE 仍默认 cpu。")


## 协调方以后要跑真实实验时

1. 工作目录设为仓库根。需要时装依赖：`pip install -r requirements-fair.txt --extra-index-url https://download.pytorch.org/whl/cpu`。GPU 机器改用对应 CUDA wheel，精度只用 bf16。
2. 浏览器里打开 Colab，第一格打印 hostname。挂载 Drive 的格子如果要开，先把 `MOUNT_DRIVE` 改成 True，挂完再打印一次 hostname，两次必须相同。
3. 把合法取得的权重路径写入 `WEIGHT_PATH`，把你在本地算出的 sha256 写入 `EXPECTED_SHA`。InnerLogic ViT-B/16 的公开 pin 是 `a516a775…`。通用 DINOv3 需先接受许可证；没有本地 sha 会拒绝加载，也不能拿 InnerLogic 文件顶替。`surgfm_probe` 要在 PYTHONPATH 上，本入口不下载权重。
4. `DATA_ROOT` 指向图像根目录，`MANIFEST` 是 jsonl（image、mask、cls、patient、split）。split 需要 train 和 val。test 不参加选点。患者 ID 只写在本地输出里。
5. 选点固定为 val 上的帧等权 binary foreground Dice。磁盘不足、目录不可写、权重缺失会直接停下。结果只写到新的 `fairrun-` 目录。不要写进 `results_e` 或 `viz_ep10`。不要从 ep10 续训。
6. 名为 `best_control.pt`、binary val 为 0.8682456 的文件会触发历史 B2 警告。它不是身份。control head 没有 backbone_state，full finetune 才有。
7. 自动测试没有在真实 Colab 或 GPU 上跑过这四组。